In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os
import re

# Configure the plotting routines
import pandas as pd

# Import your CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "password"   # change only if your real password is different

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Read all records for initial unfiltered table
df = pd.DataFrame.from_records(db.read({}))

# Remove Mongo _id because Dash DataTable does not like ObjectId
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    html.Div([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image),
                style={'height': '120px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H1('Joshua Kelly - SNHU CS-340 Dashboard')
    ], style={'textAlign': 'center'}),

    html.Hr(),

    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='all',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ], style={'margin-bottom': '20px'}),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

def breed_regex_query(breeds):
    return {'$regex': '|'.join([re.escape(b) for b in breeds]), '$options': 'i'}

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': breed_regex_query([
                'Labrador Retriever',
                'Chesapeake Bay Retriever',
                'Newfoundland'
            ]),
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': breed_regex_query([
                'German Shepherd',
                'Alaskan Malamute',
                'Old English Sheepdog',
                'Siberian Husky',
                'Rottweiler'
            ]),
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': breed_regex_query([
                'Doberman Pinscher',
                'German Shepherd',
                'Golden Retriever',
                'Bloodhound',
                'Rottweiler'
            ]),
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if dff.empty:
        return []

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution'
    )

    return [dcc.Graph(figure=fig)]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[dff.loc[row, 'location_lat'], dff.loc[row, 'location_long']],
                    children=[
                        dl.Tooltip(str(dff.loc[row, 'breed'])),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(str(dff.loc[row, 'name']))
                        ])
                    ]
                )
            ]
        )
    ]

# Run app
app.run_server()

Dash app running on https://lifeedgar-palmaappear-3000.codio.io/proxy/8050/
